# Case study: Alzheimer's disease in AlzKG

Mirrors `../../PrimeKG/case_study/autism.ipynb`'s pattern: search for the disease
across the source ontologies, then walk the built graph (`ad_kg.csv`) outward from
the Alzheimer's disease node to known drugs, genes, and variants, as a sanity
check that `../knowledge_graph/build_ad_kg.ipynb` produced a sensible graph.

In [ ]:
import numpy as np
import pandas as pd

primekg_data = '../../PrimeKG/datasets/data/'
ad_data = '../datasets/data/'

## Search for "alzheimer" across the source vocabularies

In [ ]:
dfs = []
dfs.append(('mondo', 'name', primekg_data + 'mondo/mondo_terms.csv'))
dfs.append(('umls', 'source_name', primekg_data + 'umls/umls.csv'))
dfs.append(('disgenet_ad', 'diseaseName', ad_data + 'disgenet/disgenet_ad_gene_disease.csv'))

for name, col, pth in dfs:
    try:
        df = pd.read_csv(pth, low_memory=False)
    except FileNotFoundError:
        print(f'{name}: file not found ({pth}), skipping')
        continue
    x = df.get([col]).dropna().query('{}.str.contains("alzheimer", case=False)'.format(col)).drop_duplicates()
    print(name, x.shape[0])
    display(x.head(10))

## Alzheimer's disease in AlzKG

In [ ]:
ad_kg = pd.read_csv(ad_data + 'kg/ad_kg.csv', low_memory=False)
ad_kg.shape

In [ ]:
AD_NAME = "Alzheimer's disease"

# All direct (one-hop) neighbors of the AD disease node, by relation type.
one_hop = ad_kg.query('x_name == @AD_NAME or y_name == @AD_NAME')
one_hop.groupby('relation').size().sort_values(ascending=False)

In [ ]:
# Known cholinesterase-inhibitor / NMDA-antagonist AD drugs, if present via
# ClinicalTrials.gov / Drug Central / SIDER edges pulled in from PrimeKG.
AD_DRUGS = ['Donepezil', 'Rivastigmine', 'Galantamine', 'Memantine', 'Lecanemab', 'Aducanumab']
ad_kg.query('x_name == @AD_NAME and y_name in @AD_DRUGS')

In [ ]:
# Known AD risk/causal genes (APOE = strongest late-onset risk allele carrier gene;
# APP/PSEN1/PSEN2 = early-onset familial AD genes).
AD_GENES = ['APOE', 'APP', 'PSEN1', 'PSEN2', 'TREM2', 'CLU', 'BIN1']
ad_kg.query('x_name == @AD_NAME and y_name in @AD_GENES and y_type == "gene/protein"')

In [ ]:
# Two-hop: drugs that target a gene that is one hop from AD (candidate repurposing
# leads), mirroring the Risperidone-via-target lookup pattern in autism.ipynb.
ad_genes_idx = ad_kg.query('x_name == @AD_NAME and y_type == "gene/protein"').get(['y_index']).drop_duplicates()
candidate_drugs = ad_kg.query('x_index in @ad_genes_idx.y_index and x_type == "gene/protein" '
                               'and y_type == "drug/chemical/compound"')
candidate_drugs.get(['x_name', 'relation', 'y_name', 'y_source']).drop_duplicates().head(20)